In [1]:
using Pkg
Pkg.instantiate()
Pkg.update()

    Updating registry at `~/.julia/registries/General.toml`
    Updating git-repo `https://github.com/euriqa-brassboard/MSSim.jl.git`
     Project No packages added to or removed from `~/projects/yyc-data/euriqa/calculations/rydberg_czs/Project.toml`
    Manifest No packages added to or removed from `~/projects/yyc-data/euriqa/calculations/rydberg_czs/Manifest.toml`
        Info We haven't cleaned this depot up for a bit, running Pkg.gc()...
      Active manifest files: 9 found
      Active artifact files: 1 found
      Active scratchspaces: 0 found
     Deleted no artifacts, repos, packages or scratchspaces


In [2]:
include("sqrt_cz.jl")

opt_n! (generic function with 1 method)

In [3]:
using NPZ

In [4]:
Ω = 2π * 3
nseg = 30
nsubsample = 30
t_gate = 0.7
opt = SplineMultiOpt(Ω=Ω, nseg=nseg, nsubsample=nsubsample, t_gate=t_gate,
                     δωs=(-1.7, -0.6, 0.0, 0.6, 1.7),
                     weights=(0.02, 0.3, 1.0, 0.3, 0.02),
                     lam_robs=(0.0, 0.0001, 0.2, 0.0001, 0.0),
                     lam_leaks=(0.0, 0.1, 1.0, 0.1, 0.0) .* 1.5,
                     lam_darks=(0.0, 0.0, 1.0, 0.0, 0.0) .* 0.2,
                     maxtime=5.0, maxeval_pre=1000);
# opt = Opt(Ω=Ω, num_slices=num_slices, t_gate=t_gate,
#           lam_rob=0.1, lam_leak=1, lam_dark=1);
# optional keyword arguments:
# algorithm=:LD_CCSAQ, maxeval_pre=1000, maxtime=3, xtol=1e-7, minω=-2π * 10, maxω=2π * 10

In [5]:
best_obj, best_args = @time opt_n!(opt, 40; verbose=true, pre_threshold=0.01) # default verbosity is true

  2.160471 seconds (2.33 M allocations: 115.428 MiB, 1.98% gc time, 75.72% compilation time)
obj = 0.01022170621093779
  0.358821 seconds (12 allocations: 320 bytes)
  0.352853 seconds (12 allocations: 320 bytes)
  0.357646 seconds (12 allocations: 320 bytes)
  0.359282 seconds (12 allocations: 320 bytes)
  5.356937 seconds (24 allocations: 640 bytes)
obj = 0.005664243789265382
  0.359913 seconds (12 allocations: 320 bytes)
  0.351320 seconds (12 allocations: 320 bytes)
  0.357878 seconds (12 allocations: 320 bytes)
  5.358160 seconds (24 allocations: 640 bytes)
  5.359230 seconds (24 allocations: 640 bytes)
obj = 0.004077334581116943
  0.355737 seconds (12 allocations: 320 bytes)
  5.356823 seconds (24 allocations: 640 bytes)
  5.359730 seconds (24 allocations: 640 bytes)
  5.359438 seconds (24 allocations: 640 bytes)
  0.356792 seconds (12 allocations: 320 bytes)
  0.356726 seconds (12 allocations: 320 bytes)
  0.356469 seconds (12 allocations: 320 bytes)
  5.358669 seconds (24 alloc

(0.004077334581116943, [-5.500868156485068, 7.002401383283222, 5.601759333531271, 33.58248701985361, 18.84129643948889, 41.19524728067302, 13.736093521574583, 23.971139118726295, 10.496988854756978, -32.1838876295876  …  32.92548253875108, 52.78098686185224, 54.72380104016484, 22.0722288605808, 37.55749823463289, 62.831853071795855, 62.83185307179586, 62.83185307179586, 62.83185307179586, -10.739028288831399])

In [ ]:
# More tries to refine the result.
for _ in 1:25
    best_obj, best_args = @time opt_n!(opt, 40; pre_threshold=0.01,
                                       verbose=true, best_obj=best_obj, best_args=best_args)
    if best_obj < 0.005
        break
    end
end

  5.357448 seconds (27 allocations: 752 bytes)
  5.359453 seconds (24 allocations: 640 bytes)
  0.353224 seconds (12 allocations: 320 bytes)
  0.352528 seconds (12 allocations: 320 bytes)
  0.346742 seconds (12 allocations: 320 bytes)
  0.358262 seconds (12 allocations: 320 bytes)
  0.356410 seconds (12 allocations: 320 bytes)
  0.360460 seconds (12 allocations: 320 bytes)
  5.359017 seconds (24 allocations: 640 bytes)
  0.359120 seconds (12 allocations: 320 bytes)


In [ ]:
best_ϕs = fm_to_phase(opt, best_args)
println(best_ϕs)

In [ ]:
npzwrite("sqrtcz_0.6us_3MHz.npz", Dict("phase_list"=>best_ϕs, "t_gate"=>t_gate, "Omega"=>Ω))